# PneumoFusionNet — Phase 2: Multimodal Fusion
## Image Features (ResNet50+GCSA) + Text Features (ClinicalBERT) → Binary Pneumonia Detection

### Architecture
```
CXR Image  -->  [Phase 1: ResNet50+DSC+GCSA]  -->  Image Features (1024-d)
                                                          |
                                                     [Concat]
                                                          |
Report Text --> [ClinicalBERT Encoder]          -->  Text Features  (768-d)
                                                          |
                                               [Fusion MLP Classifier]
                                                          |
                                               Binary Output (Normal / Pneumonia)
```

### Phase 2 Design Decisions
- **Text encoder**: `emilyalsentzer/Bio_ClinicalBERT` — pretrained on MIMIC clinical notes
- **Image encoder**: Frozen Phase 1 model weights (best checkpoint)
- **Fusion**: Concatenate image (1024-d) + text (768-d) = 1792-d → MLP
- **Same splits**: Exact same train/val/test split as Phase 1 (SEED=42)
- **Text**: IMPRESSION section from radiology reports
---

In [ ]:
import os, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import torchvision.models as models

from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score, auc
)
from collections import Counter

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
BASE_DIR     = r'C:\2026\PneumoFusionNet\mimic\mimic_pilot_139'
CSV_PATH     = os.path.join(BASE_DIR, 'dataset_139', 'mimic_multimodal_dataset_v2.csv')
PHASE1_MODEL = os.path.join(BASE_DIR, 'outputs', 'best_pneumofusion_mimic.pth')
SAVE_DIR     = os.path.join(BASE_DIR, 'outputs_phase2')
MODEL_PATH   = os.path.join(SAVE_DIR, 'best_multimodal_mimic.pth')

OLD_IMG = r'C:\2026\project\mimic\MIMIC_CXR_JPG_P1\p10'
NEW_IMG = r'C:\2026\PneumoFusionNet\mimic\1000_dataset\restricted_Dowloaded_dataset\CXR_JPG_1000'

IMG_SIZE      = 224
BATCH_SIZE    = 8
NUM_EPOCHS    = 40
WEIGHT_DECAY  = 1e-4
NUM_CLASSES   = 2
WARMUP_EPOCHS = 3
GRAD_CLIP     = 1.0
MAX_TEXT_LEN  = 128
BERT_MODEL    = 'emilyalsentzer/Bio_ClinicalBERT'
CLASSES       = ['NORMAL', 'PNEUMONIA']

os.makedirs(SAVE_DIR, exist_ok=True)
print('CSV exists      :', os.path.exists(CSV_PATH))
print('Phase1 exists   :', os.path.exists(PHASE1_MODEL))
print('Save dir        :', SAVE_DIR)

## Step 1: Load Multimodal Data

In [ ]:
df = pd.read_csv(CSV_PATH)
df['image_path'] = df['image_path'].str.replace(OLD_IMG, NEW_IMG, regex=False)
df['impression'] = df['impression'].fillna('No findings reported.')

img_found = df['image_path'].apply(os.path.exists).sum()
print(f'Total          : {len(df)}')
print(f'Images found   : {img_found} / {len(df)}')
print(f'Has impression : {(df["impression"].str.strip() != "").sum()} / {len(df)}')
print(f'\nClass distribution:')
print(df['label_name'].value_counts())
print('\nSample impressions:')
for _, row in df.head(3).iterrows():
    print(f'  [{row["label_name"]}] {str(row["impression"])[:90]}')

## Step 2: Same Train/Val/Test Split as Phase 1 (SEED=42)

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.30, random_state=SEED, stratify=df['label'])
val_df,   test_df = train_test_split(temp_df, test_size=0.50, random_state=SEED, stratify=temp_df['label'])
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print('=' * 65)
for name, split in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    np_ = split['label'].sum()
    nn_ = (split['label'] == 0).sum()
    print(f'{name:6s} | Total: {len(split):3d} | Normal: {nn_:3d} | Pneumonia: {np_:3d}')
print('=' * 65)

## Step 3: Load ClinicalBERT Tokenizer

> **Why ClinicalBERT?**  
> `emilyalsentzer/Bio_ClinicalBERT` is pretrained on MIMIC-III clinical notes —  
> the same hospital system as MIMIC-CXR. It understands medical abbreviations  
> like *AP*, *PA*, *bilateral*, *consolidation* better than general BERT.

In [ ]:
print(f'Loading tokenizer: {BERT_MODEL}')
tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
test_enc  = tokenizer('No acute cardiopulmonary process.', return_tensors='pt',
                      max_length=MAX_TEXT_LEN, truncation=True)
print(f'Tokenizer loaded. Sample token count: {test_enc["input_ids"].shape[1]}')

## Step 4: Rebuild Phase 1 Image Encoder

In [ ]:
class GCSA(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.avg_pool     = nn.AdaptiveAvgPool2d(1)
        self.max_pool     = nn.AdaptiveMaxPool2d(1)
        self.mlp          = nn.Sequential(
            nn.Linear(channels, channels // reduction),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels)
        )
        self.sigmoid      = nn.Sigmoid()
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3)

    def forward(self, x):
        B, C, H, W = x.shape
        avg = self.avg_pool(x).view(B, C)
        mx  = self.max_pool(x).view(B, C)
        x   = x * self.sigmoid(self.mlp(avg) + self.mlp(mx)).view(B, C, 1, 1)
        sp  = torch.cat([x.mean(1, keepdim=True), x.max(1, keepdim=True)[0]], dim=1)
        return x * self.sigmoid(self.conv_spatial(sp))


class DSC(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.dw = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch, bias=False)
        self.pw = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return F.relu(self.bn(self.pw(self.dw(x))), inplace=True)


class PneumoFusionNet(nn.Module):
    def __init__(self, num_classes=2, freeze_until=7):
        super().__init__()
        resnet       = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        orig         = resnet.conv1
        resnet.conv1 = nn.Conv2d(1, 64, 7, stride=2, padding=3, bias=False)
        with torch.no_grad():
            resnet.conv1.weight = nn.Parameter(orig.weight.mean(dim=1, keepdim=True))
        self.backbone   = nn.Sequential(*list(resnet.children())[:-2])
        for i, c in enumerate(self.backbone.children()):
            if i < freeze_until:
                for p in c.parameters(): p.requires_grad = False
        self.dsc        = DSC(2048, 1024)
        self.gcsa       = GCSA(1024)
        self.pool       = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(1024, 512),
            nn.ReLU(inplace=True), nn.Dropout(0.3), nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.pool(self.gcsa(self.dsc(self.backbone(x)))).flatten(1))

    def get_features(self, x):
        return self.pool(self.gcsa(self.dsc(self.backbone(x)))).flatten(1)  # [B, 1024]


image_encoder = PneumoFusionNet().to(DEVICE)
image_encoder.load_state_dict(torch.load(PHASE1_MODEL, map_location=DEVICE))
for p in image_encoder.parameters():
    p.requires_grad = False
image_encoder.eval()

print('Phase 1 image encoder loaded and FROZEN.')
with torch.no_grad():
    dummy = torch.randn(2, 1, 224, 224).to(DEVICE)
    print(f'Image feature shape: {image_encoder.get_features(dummy).shape}')  # [2, 1024]

## Step 5: Multimodal Dataset & DataLoaders

In [ ]:
MEAN = [0.482]
STD  = [0.220]

img_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD)
])


class MultimodalDataset(Dataset):
    """
    Returns: image [1,224,224], input_ids [max_len], attention_mask [max_len], label int
    """
    def __init__(self, df, tokenizer, max_len=MAX_TEXT_LEN, transform=None):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(row['image_path'])
        if self.transform: img = self.transform(img)
        text = str(row['impression']) if pd.notna(row['impression']) else 'No findings.'
        enc  = self.tokenizer(text, max_length=self.max_len, padding='max_length',
                              truncation=True, return_tensors='pt')
        return (img,
                enc['input_ids'].squeeze(0),
                enc['attention_mask'].squeeze(0),
                int(row['label']))


# Weighted sampler
train_labels   = train_df['label'].values
class_counts   = np.bincount(train_labels)
sample_weights = [1.0 / class_counts[l] for l in train_labels]
sampler        = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

dataloaders = {
    'train': DataLoader(MultimodalDataset(train_df, tokenizer, transform=img_transform),
                        batch_size=BATCH_SIZE, sampler=sampler, num_workers=0),
    'val'  : DataLoader(MultimodalDataset(val_df,   tokenizer, transform=img_transform),
                        batch_size=BATCH_SIZE, shuffle=False,   num_workers=0),
    'test' : DataLoader(MultimodalDataset(test_df,  tokenizer, transform=img_transform),
                        batch_size=BATCH_SIZE, shuffle=False,   num_workers=0),
}

# Verify
img, ids, mask, lbl = next(iter(dataloaders['train']))
print(f'Batch - Image: {img.shape} | IDs: {ids.shape} | Mask: {mask.shape} | Labels: {lbl}')

## Step 6: Multimodal Fusion Model

```
Image  -->  [Frozen Phase 1]  -->  1024-d
Text   -->  [ClinicalBERT]    -->   768-d
                     [Concat: 1792-d]
                [LayerNorm → 512 → 256 → 2]
```

In [ ]:
class MultimodalFusionNet(nn.Module):
    """
    Fuses frozen image features (1024-d) with ClinicalBERT CLS token (768-d)
    via concat + MLP. Only BERT + fusion MLP are trained.
    """
    def __init__(self, bert_name, img_dim=1024, txt_dim=768, num_classes=2):
        super().__init__()
        self.bert  = AutoModel.from_pretrained(bert_name)
        fused_dim  = img_dim + txt_dim  # 1792
        self.fusion = nn.Sequential(
            nn.LayerNorm(fused_dim),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def encode_text(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state[:, 0, :]  # [B, 768] CLS token

    def forward(self, img_feats, input_ids, attn_mask):
        txt_feats = self.encode_text(input_ids, attn_mask)      # [B, 768]
        fused     = torch.cat([img_feats, txt_feats], dim=1)    # [B, 1792]
        return self.fusion(fused)                                # [B, 2]


print(f'Loading ClinicalBERT: {BERT_MODEL}')
fusion_model = MultimodalFusionNet(BERT_MODEL).to(DEVICE)

total     = sum(p.numel() for p in fusion_model.parameters())
trainable = sum(p.numel() for p in fusion_model.parameters() if p.requires_grad)
print(f'Total params    : {total:,}')
print(f'Trainable params: {trainable:,} ({trainable/total*100:.1f}%)')

# Quick sanity check
with torch.no_grad():
    out = fusion_model(torch.randn(2,1024).to(DEVICE),
                       torch.randint(0,100,(2,MAX_TEXT_LEN)).to(DEVICE),
                       torch.ones(2,MAX_TEXT_LEN,dtype=torch.long).to(DEVICE))
print(f'Output shape    : {out.shape}')  # [2, 2]

## Step 7: Optimizer with Differential LR

> BERT uses `lr=1e-5` (small — avoid catastrophic forgetting)  
> Fusion MLP uses `lr=5e-4` (larger — trains from scratch)

In [ ]:
optimizer = optim.AdamW([
    {'params': fusion_model.bert.parameters(),   'lr': 1e-5},
    {'params': fusion_model.fusion.parameters(), 'lr': 5e-4},
], weight_decay=WEIGHT_DECAY)

criterion     = nn.CrossEntropyLoss()
linear_warmup = LinearLR(optimizer, start_factor=0.1, total_iters=WARMUP_EPOCHS)
cosine_anneal = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - WARMUP_EPOCHS)
scheduler     = SequentialLR(optimizer, [linear_warmup, cosine_anneal], milestones=[WARMUP_EPOCHS])

print('Optimizer    : AdamW (differential LR)')
print('  BERT       : lr=1e-5')
print('  Fusion MLP : lr=5e-4')

## Step 8: Training Loop

In [ ]:
def evaluate_mm(fusion_model, image_encoder, loader, criterion, device):
    fusion_model.eval(); image_encoder.eval()
    total_loss, all_preds, all_labels, all_probs = 0.0, [], [], []
    with torch.no_grad():
        for imgs, ids, mask, labels in loader:
            imgs, ids, mask, labels = imgs.to(device), ids.to(device), mask.to(device), labels.to(device)
            img_feats  = image_encoder.get_features(imgs)
            outputs    = fusion_model(img_feats, ids, mask)
            total_loss += criterion(outputs, labels).item() * len(labels)
            probs       = F.softmax(outputs, dim=1)[:, 1]
            all_preds.extend(outputs.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
            all_probs.extend(probs.cpu().tolist())
    avg_loss  = total_loss / len(loader.dataset)
    accuracy  = accuracy_score(all_labels, all_preds)
    auc_score = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.0
    return avg_loss, accuracy, auc_score, all_preds, all_labels, all_probs


history          = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_auc': []}
best_val_auc     = 0.0
best_epoch       = 0
patience_counter = 0
PATIENCE         = 10

print('\n' + '='*80)
print('PHASE 2 MULTIMODAL TRAINING')
print('='*80)

for epoch in range(NUM_EPOCHS):
    fusion_model.train(); image_encoder.eval()
    train_loss = train_correct = train_n = 0

    for imgs, ids, mask, labels in dataloaders['train']:
        imgs, ids, mask, labels = imgs.to(DEVICE), ids.to(DEVICE), mask.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.no_grad():
            img_feats = image_encoder.get_features(imgs)
        outputs = fusion_model(img_feats, ids, mask)
        loss    = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(fusion_model.parameters(), GRAD_CLIP)
        optimizer.step()
        train_loss    += loss.item() * len(labels)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_n       += len(labels)

    scheduler.step()
    train_loss_avg = train_loss / train_n
    train_acc      = train_correct / train_n

    val_loss, val_acc, val_auc, _, _, _ = evaluate_mm(
        fusion_model, image_encoder, dataloaders['val'], criterion, DEVICE)

    history['train_loss'].append(train_loss_avg)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['val_auc'].append(val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc; best_epoch = epoch; patience_counter = 0
        torch.save(fusion_model.state_dict(), MODEL_PATH)
        marker = ' <- BEST'
    else:
        patience_counter += 1; marker = ''

    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch [{epoch+1:2d}/{NUM_EPOCHS}] | '
              f'Train: {train_loss_avg:.4f} ({train_acc:.3f}) | '
              f'Val: {val_loss:.4f} ({val_acc:.3f}) | '
              f'AUC: {val_auc:.4f}{marker}')

    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch+1}'); break

print('='*80)
print(f'Best val AUC: {best_val_auc:.4f} at epoch {best_epoch+1}')
print(f'Saved: {MODEL_PATH}')
print('='*80)

## Step 9: Training History Plot

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Phase 2 — Multimodal Training History', fontsize=13, fontweight='bold')
axes[0].plot(history['train_loss'], label='Train', lw=2)
axes[0].plot(history['val_loss'],   label='Val',   lw=2)
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(history['val_acc'], lw=2, color='green', label='Val Acc')
axes[1].axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
axes[1].set_title('Validation Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
axes[2].plot(history['val_auc'], lw=2, color='orange', label='Val AUC')
axes[2].axvline(best_epoch, color='red', linestyle='--', alpha=0.5)
axes[2].axhline(best_val_auc, color='red', linestyle=':', alpha=0.5)
axes[2].set_title('Validation AUC'); axes[2].legend(); axes[2].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_training_history.png'), dpi=120, bbox_inches='tight')
plt.show()

## Step 10: Test Set Evaluation

In [ ]:
fusion_model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
test_loss, test_acc, test_auc, test_preds, test_labels, test_probs = evaluate_mm(
    fusion_model, image_encoder, dataloaders['test'], criterion, DEVICE)

print('\n' + '='*60)
print('PHASE 2 — TEST SET RESULTS')
print('='*60)
print(f'Loss     : {test_loss:.4f}')
print(f'Accuracy : {test_acc:.4f}')
print(f'AUC      : {test_auc:.4f}')
print('\nClassification Report:')
print(classification_report(test_labels, test_preds, target_names=CLASSES))
print('='*60)

## Step 11: Confusion Matrix & ROC

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Phase 2 — Multimodal Fusion Results', fontsize=13, fontweight='bold')
cm = confusion_matrix(test_labels, test_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].set_title('Confusion Matrix')
fpr, tpr, _ = roc_curve(test_labels, test_probs)
roc_auc_val = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='darkorange', lw=2.5, label=f'AUC = {roc_auc_val:.4f}')
axes[1].plot([0,1],[0,1],'navy',lw=1.5,linestyle='--',label='Random')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('ROC Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'phase2_results.png'), dpi=120, bbox_inches='tight')
plt.show()

## Step 12: Phase 1 vs Phase 2 Comparison
> Update `phase1_auc` and `phase1_acc` after Phase 1 training completes.

In [ ]:
# Update these after Phase 1 test results
phase1_auc = None   # e.g. 0.7812
phase1_acc = None   # e.g. 0.7143

print('=' * 65)
print('PHASE 1 vs PHASE 2 — FINAL COMPARISON')
print('=' * 65)
print(f'{"Model":<30} {"AUC":>8} {"Accuracy":>10}')
print('-' * 65)
p1a = f'{phase1_auc:.4f}' if phase1_auc else 'TBD'
p1c = f'{phase1_acc:.4f}' if phase1_acc else 'TBD'
print(f'{"Phase 1 (Image Only)":<30} {p1a:>8} {p1c:>10}')
print(f'{"Phase 2 (Image + Report)":<30} {test_auc:>8.4f} {test_acc:>10.4f}')
print('=' * 65)
if phase1_auc:
    delta = test_auc - phase1_auc
    print(f'\nAUC improvement from multimodal: {delta:+.4f}')

print(f'\nOutputs: {SAVE_DIR}')
for f in sorted(os.listdir(SAVE_DIR)):
    print(f'  - {f}')

## Summary

| Component | Detail |
|---|---|
| **Image Encoder** | Frozen Phase 1 ResNet50+DSC+GCSA → 1024-d |
| **Text Encoder** | Bio_ClinicalBERT CLS token → 768-d |
| **Fusion** | Concat 1792-d → LayerNorm → GELU MLP → 2-class |
| **Text Input** | IMPRESSION section from radiology report |
| **BERT LR** | 1e-5 (fine-tune carefully) |
| **MLP LR** | 5e-4 (train from scratch) |
| **Class Imbalance** | WeightedRandomSampler on train set |

### Next Steps
- Cross-attention fusion (image tokens attend to text tokens)
- Grad-CAM on images + attention heatmap on text tokens
- Scale to full 1000-image dataset